# Can we predict 30-day readmission?

This notebook tests whether the supplied admission/discharge variables predict `readmit_30d`. The unit of observation is a hospital stay, but patients have repeated stays. Therefore every split is grouped by `person_key`: no patient can occur in both the training and validation/test portions of a split.

**Primary metric:** ROC AUC (ranking discrimination). **Supporting metrics:** average precision (appropriate for the imbalanced outcome), Brier score (probability accuracy), calibration, and threshold-dependent classification metrics. The final test set is held out until model and threshold selection are complete.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score, balanced_accuracy_score, brier_score_loss,
    classification_report, confusion_matrix, precision_recall_curve,
    roc_auc_score, roc_curve,
)
from sklearn.model_selection import (
    StratifiedGroupKFold, cross_validate, cross_val_predict,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 50)

## 1. Load and understand the data

In [ ]:
DATA_PATH = Path('data.csv')
DICTIONARY_PATH = Path('columns.csv')

data = pd.read_csv(DATA_PATH)
dictionary = pd.read_csv(DICTIONARY_PATH)

print(f'{len(data):,} stays, {data.person_key.nunique():,} people, {data.shape[1]} columns')
display(dictionary)
display(data.head())

In [ ]:
audit = pd.DataFrame({
    'dtype': data.dtypes.astype(str),
    'missing_n': data.isna().sum(),
    'missing_pct': data.isna().mean().mul(100).round(2),
    'unique_n': data.nunique(dropna=False),
})
display(audit)

assert set(data['readmit_30d'].dropna().unique()) <= {0, 1}
assert data.duplicated().sum() == 0
assert data.duplicated(['person_key', 'visit_number']).sum() == 0

person_counts = data['person_key'].value_counts()
print(f'Readmission prevalence: {data.readmit_30d.mean():.1%}')
print(f'People with >1 stay: {(person_counts > 1).sum():,} / {len(person_counts):,}')
print(f'Stays per person: median={person_counts.median():.0f}, max={person_counts.max()}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.countplot(data=data, x='readmit_30d', ax=axes[0], color='#4C78A8')
axes[0].set(title='Outcome balance', xlabel='Readmitted within 30 days', ylabel='Stays')
numeric = data.select_dtypes(include=np.number).drop(columns=['person_key'])
corr = numeric.corr(numeric_only=True)['readmit_30d'].drop('readmit_30d').sort_values()
corr.plot.barh(ax=axes[1], color='#F58518')
axes[1].set(title='Univariate Pearson correlations', xlabel='Correlation with readmit_30d')
plt.tight_layout()
plt.show()
display(data.groupby('readmit_30d').agg(
    stays=('person_key', 'size'),
    mean_length_of_stay=('length_of_stay_days', 'mean'),
    mean_medications=('n_medications', 'mean'),
    mean_lab_a=('lab_a', 'mean'),
).round(2))

### Modeling decisions and leakage controls

- `person_key` is an identifier, not a clinical predictor. It is used only to group splits.
- `postcode_area` is encoded as categorical: numeric distances between area codes have no stated meaning.
- `sex` is categorical; remaining predictors are treated as numeric.
- All preprocessing is fitted inside each training fold through a pipeline.
- The outer test fold contains entirely unseen people and is used once, after cross-validated model selection.
- These columns mix admission (`bmi`, labs) and discharge (`length_of_stay_days`, `n_medications`) information. Results therefore describe prediction **at discharge**, not at admission.
- There is no admission/discharge date, so a temporal validation is impossible. `birth_year` is retained as a proxy, but true age cannot be derived.
- We do not create features from prior target values. Such history may be useful operationally, but constructing it safely would require timestamps and a clearly defined prediction workflow.

## 2. Patient-grouped train/test split

In [ ]:
target = 'readmit_30d'
group_column = 'person_key'
feature_columns = [c for c in data.columns if c not in [target, group_column]]
X = data[feature_columns].copy()
y = data[target].astype(int).copy()
groups = data[group_column].copy()

# One fold of a shuffled, stratified, patient-grouped 5-fold split is the 20% test set.
outer_splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
train_idx, test_idx = next(outer_splitter.split(X, y, groups))
X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()
groups_train, groups_test = groups.iloc[train_idx].copy(), groups.iloc[test_idx].copy()

assert set(groups_train).isdisjoint(set(groups_test))
split_summary = pd.DataFrame({
    'stays': [len(train_idx), len(test_idx)],
    'people': [groups_train.nunique(), groups_test.nunique()],
    'readmission_rate': [y_train.mean(), y_test.mean()],
}, index=['train', 'test'])
display(split_summary.style.format({'readmission_rate': '{:.1%}'}))

## 3. Compare candidate models with grouped cross-validation

The dummy model estimates only the training prevalence. Logistic regression provides a regularized, interpretable linear baseline. Random forest allows nonlinearities and interactions. Model selection uses only the training set.

In [ ]:
categorical_features = ['sex', 'postcode_area']
numeric_features = [c for c in feature_columns if c not in categorical_features]

numeric_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler()),
])
categorical_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ('numeric', numeric_pipe, numeric_features),
    ('categorical', categorical_pipe, categorical_features),
])

estimators = {
    'Dummy prevalence': DummyClassifier(strategy='prior'),
    'Logistic regression': LogisticRegression(max_iter=3000),
    'Random forest': RandomForestClassifier(
        n_estimators=400, min_samples_leaf=20, max_features='sqrt',
        n_jobs=-1, random_state=RANDOM_STATE,
    ),
}
models = {
    name: Pipeline([('preprocess', clone(preprocessor)), ('model', estimator)])
    for name, estimator in estimators.items()
}
scoring = {
    'roc_auc': 'roc_auc',
    'average_precision': 'average_precision',
    'neg_brier': 'neg_brier_score',
}
inner_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

In [ ]:
cv_rows = []
for name, model in models.items():
    scores = cross_validate(
        model, X_train, y_train, groups=groups_train, cv=inner_cv,
        scoring=scoring, n_jobs=-1, return_train_score=False,
    )
    cv_rows.append({
        'model': name,
        'roc_auc_mean': scores['test_roc_auc'].mean(),
        'roc_auc_sd': scores['test_roc_auc'].std(ddof=1),
        'average_precision_mean': scores['test_average_precision'].mean(),
        'average_precision_sd': scores['test_average_precision'].std(ddof=1),
        'brier_mean': -scores['test_neg_brier'].mean(),
    })
cv_results = pd.DataFrame(cv_rows).sort_values('roc_auc_mean', ascending=False).reset_index(drop=True)
display(cv_results.style.format({c: '{:.3f}' for c in cv_results.columns if c != 'model'}))

best_name = cv_results.loc[0, 'model']
best_model = clone(models[best_name])
print(f'Selected by mean grouped-CV ROC AUC: {best_name}')

## 4. Select an illustrative operating threshold without using the test set

A clinical threshold should ultimately reflect the relative cost, capacity, and benefit of interventions. Because those are unspecified, the threshold below is only illustrative: it maximizes Youden's J statistic (equivalently, sensitivity + specificity - 1) on out-of-fold training predictions.

In [ ]:
oof_probability = cross_val_predict(
    clone(best_model), X_train, y_train, groups=groups_train, cv=inner_cv,
    method='predict_proba', n_jobs=-1,
)[:, 1]
fpr_train, tpr_train, thresholds_train = roc_curve(y_train, oof_probability)
finite = np.isfinite(thresholds_train)
best_threshold = thresholds_train[finite][np.argmax((tpr_train - fpr_train)[finite])]
print(f'OOF-selected illustrative threshold: {best_threshold:.3f}')

## 5. Final evaluation on unseen patients

In [ ]:
best_model.fit(X_train, y_train)
test_probability = best_model.predict_proba(X_test)[:, 1]
test_prediction = (test_probability >= best_threshold).astype(int)

point_metrics = {
    'ROC AUC': roc_auc_score(y_test, test_probability),
    'Average precision': average_precision_score(y_test, test_probability),
    'Brier score': brier_score_loss(y_test, test_probability),
    'Balanced accuracy @ selected threshold': balanced_accuracy_score(y_test, test_prediction),
}
display(pd.Series(point_metrics, name='test').to_frame().style.format('{:.3f}'))
print(classification_report(y_test, test_prediction, target_names=['not readmitted', 'readmitted'], digits=3))

In [ ]:
def grouped_bootstrap_interval(y_true, probability, group_ids, metric, n_boot=2000, seed=42):
    """Percentile CI resampling people, preserving all stays from sampled people."""
    frame = pd.DataFrame({'y': np.asarray(y_true), 'p': probability, 'group': np.asarray(group_ids)})
    by_group = {key: value[['y', 'p']].to_numpy() for key, value in frame.groupby('group')}
    unique_groups = np.array(list(by_group))
    rng = np.random.default_rng(seed)
    values = []
    for _ in range(n_boot):
        sampled = rng.choice(unique_groups, size=len(unique_groups), replace=True)
        sample = np.concatenate([by_group[g] for g in sampled])
        if np.unique(sample[:, 0]).size == 2:
            values.append(metric(sample[:, 0], sample[:, 1]))
    return np.quantile(values, [0.025, 0.975])

roc_ci = grouped_bootstrap_interval(y_test, test_probability, groups_test, roc_auc_score)
ap_ci = grouped_bootstrap_interval(y_test, test_probability, groups_test, average_precision_score)
ci_table = pd.DataFrame({
    'estimate': [point_metrics['ROC AUC'], point_metrics['Average precision']],
    '95% CI low': [roc_ci[0], ap_ci[0]],
    '95% CI high': [roc_ci[1], ap_ci[1]],
}, index=['ROC AUC', 'Average precision'])
display(ci_table.style.format('{:.3f}'))

In [ ]:
from sklearn.calibration import calibration_curve

fpr, tpr, _ = roc_curve(y_test, test_probability)
precision, recall, _ = precision_recall_curve(y_test, test_probability)
fraction_positive, mean_predicted = calibration_curve(y_test, test_probability, n_bins=8, strategy='quantile')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(fpr, tpr, color='#4C78A8', label=f"AUC = {point_metrics['ROC AUC']:.3f}")
axes[0].plot([0, 1], [0, 1], '--', color='grey')
axes[0].set(xlabel='False-positive rate', ylabel='True-positive rate', title='ROC curve')
axes[0].legend()
axes[1].plot(recall, precision, color='#F58518', label=f"AP = {point_metrics['Average precision']:.3f}")
axes[1].axhline(y_test.mean(), ls='--', color='grey', label=f'Prevalence = {y_test.mean():.3f}')
axes[1].set(xlabel='Recall', ylabel='Precision', title='Precision-recall curve')
axes[1].legend()
axes[2].plot(mean_predicted, fraction_positive, marker='o', color='#54A24B')
axes[2].plot([0, 1], [0, 1], '--', color='grey')
axes[2].set(xlabel='Mean predicted probability', ylabel='Observed fraction', title='Calibration')
plt.tight_layout()
plt.show()

cm = confusion_matrix(y_test, test_prediction)
display(pd.DataFrame(cm, index=['actual 0', 'actual 1'], columns=['predicted 0', 'predicted 1']))

## 6. What is driving predictions?

Permutation importance measures the drop in test ROC AUC when one raw feature is shuffled. It captures model reliance rather than causality; correlated predictors can share importance.

In [ ]:
importance = permutation_importance(
    best_model, X_test, y_test, scoring='roc_auc', n_repeats=20,
    random_state=RANDOM_STATE, n_jobs=-1,
)
importance_table = pd.DataFrame({
    'feature': X_test.columns,
    'importance_mean': importance.importances_mean,
    'importance_sd': importance.importances_std,
}).sort_values('importance_mean', ascending=False)
display(importance_table.style.format({'importance_mean': '{:.3f}', 'importance_sd': '{:.3f}'}))

plot_data = importance_table.sort_values('importance_mean')
plt.figure(figsize=(8, 5))
plt.barh(plot_data['feature'], plot_data['importance_mean'], xerr=plot_data['importance_sd'], color='#4C78A8')
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Decrease in ROC AUC after permutation')
plt.title('Held-out permutation importance')
plt.tight_layout()
plt.show()

## 7. Basic subgroup diagnostic

This is not a full fairness assessment. It checks whether discrimination and error rates differ visibly by recorded sex; estimates may be noisy and do not establish equity or clinical safety.

In [ ]:
subgroup_rows = []
for value in sorted(X_test['sex'].dropna().unique()):
    mask = X_test['sex'].eq(value).to_numpy()
    yt, pt, pred = y_test.iloc[np.flatnonzero(mask)], test_probability[mask], test_prediction[mask]
    tn, fp, fn, tp = confusion_matrix(yt, pred, labels=[0, 1]).ravel()
    subgroup_rows.append({
        'sex': value, 'n': mask.sum(), 'prevalence': yt.mean(),
        'roc_auc': roc_auc_score(yt, pt),
        'average_precision': average_precision_score(yt, pt),
        'sensitivity': tp / (tp + fn), 'specificity': tn / (tn + fp),
    })
subgroups = pd.DataFrame(subgroup_rows).set_index('sex')
display(subgroups.style.format({
    'prevalence': '{:.1%}', 'roc_auc': '{:.3f}', 'average_precision': '{:.3f}',
    'sensitivity': '{:.1%}', 'specificity': '{:.1%}',
}))

## 8. Conclusion

Use the cell below as the evidence-based answer. A model can be statistically better than a prevalence-only baseline while still being too weak for clinical use. This experiment establishes internal predictive signal—not prospective clinical utility.

In [ ]:
auc = point_metrics['ROC AUC']
ap = point_metrics['Average precision']
if auc >= 0.80:
    strength = 'strong'
elif auc >= 0.70:
    strength = 'moderate'
elif auc >= 0.60:
    strength = 'modest'
else:
    strength = 'weak'

print(
    f'Answer: the supplied variables contain {strength} predictive signal for 30-day readmission.\n'
    f'The selected {best_name} achieved held-out patient-grouped ROC AUC {auc:.3f} '
    f'(95% grouped-bootstrap CI {roc_ci[0]:.3f}–{roc_ci[1]:.3f}) and average precision {ap:.3f} '
    f'against a test prevalence baseline of {y_test.mean():.3f}.\n'
    'This is evidence that prediction is possible above chance, but the discrimination is not strong enough by itself '
    'to justify clinical deployment. Before deployment, add clinically relevant history and diagnosis/comorbidity features, '
    'validate on a later time period and external site, assess calibration and subgroup performance, and choose a threshold '
    'using intervention costs and capacity.'
)